# Open-Vocabulary Vision — CLIP Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: A tiny two-tower model

Real CLIP is ViT + transformer. For this lesson the towers are small MLPs over pre-extracted features so the training signal is visible on CPU.

In [ ]:
```python

import torch

import torch.nn as nn

import torch.nn.functional as F

class TwoTower(nn.Module):

    def __init__(self, img_in=128, txt_in=64, emb=64):

        super().__init__()

        self.image_proj = nn.Sequential(nn.Linear(img_in, 128), nn.ReLU(), nn.Linear(128, emb))

        self.text_proj = nn.Sequential(nn.Linear(txt_in, 128), nn.ReLU(), nn.Linear(128, emb))

        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)  # ln(1/0.07)

    def forward(self, img_feats, txt_feats):

        i = F.normalize(self.image_proj(img_feats), dim=-1)

        t = F.normalize(self.text_proj(txt_feats), dim=-1)

        return i, t, self.logit_scale.exp()

In [ ]:
```

Two projections, shared-dim output, learned temperature. Same shape as the real CLIP API.

### Step 2: Contrastive loss

In [ ]:
```python

def clip_loss(image_emb, text_emb, logit_scale):

    N = image_emb.size(0)

    sim = logit_scale * image_emb @ text_emb.T

    targets = torch.arange(N, device=sim.device)

    l_i = F.cross_entropy(sim, targets)

    l_t = F.cross_entropy(sim.T, targets)

    return (l_i + l_t) / 2

In [ ]:
```

Symmetric. Higher logit_scale = sharper softmax = more confident but risk of instability.

### Step 3: Zero-shot classifier

In [ ]:
```python

@torch.no_grad()

def zero_shot_classify(model, image_feats, class_text_feats, class_names):

    """

    image_feats:      (N, img_in)

    class_text_feats: (C, txt_in)   one averaged embedding per class

    """

    i = F.normalize(model.image_proj(image_feats), dim=-1)

    t = F.normalize(model.text_proj(class_text_feats), dim=-1)

    sim = i @ t.T

    pred = sim.argmax(dim=-1)

    return [class_names[p] for p in pred.tolist()]

In [ ]:
```

One line per step. This is the exact zero-shot procedure used with a production CLIP checkpoint.

### Step 4: Sanity check

In [ ]:
```python

torch.manual_seed(0)

model = TwoTower()

img = torch.randn(8, 128)

txt = torch.randn(8, 64)

i, t, scale = model(img, txt)

loss = clip_loss(i, t, scale)

print(f"batch size: {i.size(0)}   loss: {loss.item():.3f}")

In [ ]:
```

Loss should be close to `log(N) = log(8) = 2.08` for a randomly initialised model — the symmetric cross-entropy target when no structure is learned yet.

## Exercises

In [ ]:
1. **(Easy)** Use a pretrained OpenCLIP ViT-B/32 and do zero-shot classification on CIFAR-10 with the 80-template prompt set. Report top-1 accuracy; it should be around 85-90%.
2. **(Medium)** Compare single-template ("a photo of a {}") vs 80-template averaged embeddings on the same CIFAR-10 task. Quantify the gap and explain why templates help.
3. **(Hard)** Build a zero-shot image retrieval index: embed 1,000 images with CLIP, build a FAISS index, query with a natural language description. Report retrieval recall@5 for 20 held-out queries you write by hand.